# SMILES to hash, from scratch

A stripped-down version of the CSH pipeline: no torsions, no canonical atom
order, no topological filter. Just the mechanics, so the shape of the thing is
visible.

Seven steps. Each one is a function with an empty body and a test underneath.
Fill in the body until the test prints PASS, then move on. The tests tell you
what went wrong, not just that something did.

Rough timing: 30 to 60 minutes if you do not look things up too much.


In [1]:
# ── Setup ────────────────────────────────────────────────────────────────────
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
RDLogger.DisableLog("rdApp.*")

ASPIRIN   = "CC(=O)Oc1ccccc1C(=O)O"
IBUPROFEN = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"

def check(name, ok, detail=""):
    """Print one test result; the hint is only shown when the test fails."""
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + ("" if ok or not detail else f"\n        {detail}"))
    return ok

print("ready")

ready


## 1. SMILES to conformers

`make_conformers(smiles, n, seed)` should return an RDKit molecule carrying
`n` 3D conformers, energy-minimised.

Four calls, in order: parse the SMILES, add hydrogens, embed the conformers,
minimise them. Look up `AllChem.ETKDGv3`, `EmbedMultipleConfs`,
`MMFFOptimizeMoleculeConfs`.

The seed has to be set on the ETKDG parameter object, not passed to the embed
call.

In [5]:
def make_conformers(smiles, n, seed=42):
    """Return a molecule with n minimised 3D conformers, deterministic given seed."""
    m = Chem.AddHs(Chem.MolFromSmiles(smiles))
    p = AllChem.ETKDGv3()
    p.randomSeed = seed
    AllChem.EmbedMultipleConfs(m, numConfs=n, params=p)
    AllChem.MMFFOptimizeMoleculeConfs(m)
    return m
    raise NotImplementedError


# ── test ──
try:
    m = make_conformers(ASPIRIN, 5)
    ok  = check("returns a molecule", m is not None)
    ok &= check("has 5 conformers", m.GetNumConformers() == 5,
                f"got {m.GetNumConformers()}")
    ok &= check("hydrogens were added", any(a.GetSymbol() == "H" for a in m.GetAtoms()),
                "did you call Chem.AddHs?")
    a = make_conformers(ASPIRIN, 3, seed=1)
    b = make_conformers(ASPIRIN, 3, seed=1)
    same = np.allclose(a.GetConformer(0).GetPositions(), b.GetConformer(0).GetPositions())
    ok &= check("same seed gives same geometry", same,
                "is the seed reaching params.randomSeed?")
except NotImplementedError:
    print("not implemented yet")

PASS  returns a molecule
PASS  has 5 conformers
PASS  hydrogens were added
PASS  same seed gives same geometry


In [8]:
for n in (1, 3, 300):
    print(n, "->", make_conformers(ASPIRIN, n).GetNumConformers())

1 -> 1
3 -> 3
300 -> 300


## 2. Feature vector

`features(mol, conf_id)` should return the distances between every pair of
heavy atoms, as a 1-D array.

Two parts: drop the hydrogens, then compute the distances. `Chem.RemoveHs`
does the first. For the second, `pdist` on the coordinate array gives you every
pair exactly once, which is what you want.

For a molecule with N heavy atoms the result has N(N-1)/2 values.

In [25]:
def features(mol, conf_id=0):
    """Distances between all pairs of heavy atoms, as a flat array."""
    molecule = Chem.RemoveHs(mol)
    pos = molecule.GetConformer(conf_id).GetPositions()
    return pdist(pos)


# ── test ──
try:
    m = make_conformers(ASPIRIN, 2)
    v = features(m, 0)
    n_heavy = Chem.RemoveHs(m).GetNumAtoms()
    expected = n_heavy * (n_heavy - 1) // 2
    ok  = check("returns a 1-D array", np.asarray(v).ndim == 1)
    ok &= check(f"length is {expected}", len(v) == expected,
                f"got {len(v)}; aspirin has {n_heavy} heavy atoms")
    ok &= check("all distances positive", (np.asarray(v) > 0).all())
    mi = make_conformers(IBUPROFEN, 5)
    V = [tuple(np.round(features(mi, c), 9)) for c in range(mi.GetNumConformers())]
    ok &= check("different conformers give different vectors", len(set(V)) > 1,
                "all five conformers collapsed to the same geometry")
except NotImplementedError:
    print("not implemented yet")

PASS  returns a 1-D array
PASS  length is 78
PASS  all distances positive
PASS  different conformers give different vectors


## 3. Reference ensemble and scaling

The transform must not be fitted on the conformers you are going to hash, or
the result is flattered. So: build a *separate* ensemble from the same SMILES,
with a different seed, and fit the scaler on that.

`fit_scaler(smiles, n, seed)` should return a fitted `StandardScaler`.

Centring matters more than it looks. Interatomic distances are all positive and
sit a long way from zero; without subtracting the mean, every conformer ends up
on the same side of nearly every hyperplane in step 5 and all the hashes come
out identical. The test below checks that the centring actually happened.

In [26]:
REF_SEED = 7     # reference ensemble
TEST_SEED = 42   # the conformers being hashed

def fit_scaler(smiles, n=30, seed=REF_SEED):
    """Build a reference ensemble and return a StandardScaler fitted to it."""
    ref = make_conformers(ASPIRIN, 30, seed=7)
    X = np.array([features(ref, c) for c in range(ref.GetNumConformers())])
    return StandardScaler().fit(X)


# ── test ──
try:
    sc = fit_scaler(ASPIRIN)
    m = make_conformers(ASPIRIN, 5, seed=TEST_SEED)
    X = np.array([features(m, c) for c in range(5)])
    Z = sc.transform(X)
    ok  = check("returns a fitted scaler", hasattr(sc, "mean_"))
    ok &= check("one mean per feature", len(sc.mean_) == X.shape[1],
                f"got {len(sc.mean_)}, expected {X.shape[1]}")
    ok &= check("raw features are far from zero", np.abs(X).mean() > 1.0,
                f"mean |value| = {np.abs(X).mean():.2f} A")
    ok &= check("scaled features are centred near zero", np.abs(Z).mean() < 5.0,
                f"mean |value| = {np.abs(Z).mean():.2f}")
    ok &= check("reference ensemble differs from the test one",
                not np.allclose(sc.mean_, X.mean(axis=0)),
                "if these match, you fitted on the conformers you are hashing")
except NotImplementedError:
    print("not implemented yet")

PASS  returns a fitted scaler
PASS  one mean per feature
PASS  raw features are far from zero
PASS  scaled features are centred near zero
PASS  reference ensemble differs from the test one


In [27]:
sc = fit_scaler(ASPIRIN)
print("średnich:", len(sc.mean_))
print("pierwsze pięć średnich:   ", np.round(sc.mean_[:5], 3))
print("pierwsze pięć odchyleń:   ", np.round(sc.scale_[:5], 3))

średnich: 78
pierwsze pięć średnich:    [1.5   2.412 2.351 3.626 4.477]
pierwsze pięć odchyleń:    [0.    0.001 0.001 0.001 0.015]


## 4. PCA

`fit_pca(scaler, smiles, n_components, n, seed)` should return a PCA fitted to
the *scaled* reference ensemble.

Same ensemble as step 3, same seed. Scale first, then fit the PCA on the scaled
values, not the raw ones.

One catch: PCA cannot have more components than it has samples or features.
Clip `n_components` accordingly, or the call fails on small molecules.

In [32]:
def fit_pca(scaler, smiles, n_components=16, n=30, seed=REF_SEED):
    """Return a PCA fitted to the scaled reference ensemble."""
    ref = make_conformers(ASPIRIN, 30, seed=7)
    X = np.array([features(ref, c) for c in range(ref.GetNumConformers())])
    return PCA(n_components).fit(scaler.transform(X))


def reduce(scaler, pca, mol, conf_id=0):
    x = features(mol, conf_id).reshape(1, -1)
    return pca.transform(scaler.transform(x))[0]


# ── test ──
try:
    sc = fit_scaler(ASPIRIN)
    pca = fit_pca(sc, ASPIRIN, n_components=16)
    m = make_conformers(ASPIRIN, 5, seed=TEST_SEED)
    y = reduce(sc, pca, m, 0)
    ok  = check("PCA is fitted", hasattr(pca, "components_"))
    ok &= check("reduced vector has 16 values", len(y) == 16, f"got {len(y)}")
    ok &= check("reduction is deterministic",
                np.allclose(reduce(sc, pca, m, 0), reduce(sc, pca, m, 0)))
    Y = np.array([reduce(sc, pca, m, c) for c in range(5)])
    ok &= check("different conformers reduce differently",
                len({tuple(np.round(y, 9)) for y in Y}) > 1)
    var = pca.explained_variance_ratio_.sum()
    print(f"      (these 16 components keep {var:.1%} of the variance)")
except NotImplementedError:
    print("not implemented yet")

PASS  PCA is fitted
PASS  reduced vector has 16 values
PASS  reduction is deterministic
PASS  different conformers reduce differently
      (these 16 components keep 100.0% of the variance)


## 5. SimHash

`simhash(vectors, n_bits, seed)` turns real-valued vectors into bits.

Draw `n_bits` random vectors from a normal distribution, one per bit. For each
input vector, each bit says which side of that random hyperplane the vector
falls on: 1 if the dot product is positive, 0 otherwise.

Input is shape (n_conformers, n_features); output should be
(n_conformers, n_bits) of 0/1. One matrix multiplication does the whole thing,
no loop needed.

Use `np.random.RandomState(seed)` so the hyperplanes are the same every run.
That is the whole reason two people can reproduce each other's hashes.

In [35]:
def simhash(vectors, n_bits=64, seed=42):
    """Project vectors onto n_bits random hyperplanes; return 0/1 as uint8."""
    rng = np.random.RandomState(seed)
    return (vectors @ rng.randn(n_bits, vectors.shape[1]).T > 0).astype(np.uint8)


# ── test ──
try:
    sc = fit_scaler(ASPIRIN)
    pca = fit_pca(sc, ASPIRIN, n_components=16)
    m = make_conformers(ASPIRIN, 10, seed=TEST_SEED)
    Y = np.array([reduce(sc, pca, m, c) for c in range(10)])
    B = simhash(Y, n_bits=64)
    ok  = check("shape is (10, 64)", B.shape == (10, 64), f"got {B.shape}")
    ok &= check("only zeros and ones", set(np.unique(B)) <= {0, 1})
    ok &= check("roughly balanced", 0.2 < B.mean() < 0.8,
                f"fraction of ones = {B.mean():.2f}; far from 0.5 means "
                f"the input was not centred")
    ok &= check("same seed gives same bits", np.array_equal(simhash(Y), simhash(Y)))
    ok &= check("different seed gives different bits",
                not np.array_equal(simhash(Y, seed=1), simhash(Y, seed=2)))
    print(f"      ({len(set(map(tuple, B)))} distinct hashes out of 10 conformers)")
except NotImplementedError:
    print("not implemented yet")

PASS  shape is (10, 64)
PASS  only zeros and ones
PASS  roughly balanced
PASS  same seed gives same bits
PASS  different seed gives different bits
      (2 distinct hashes out of 10 conformers)


## 6. Comparing hashes

`hamming(a, b)` counts the positions where two bit arrays differ.

`similarity(a, b)` is the fraction of positions where they agree. Note which
way round each one goes: Hamming grows as the hashes diverge, similarity
shrinks. Mixing them up flips the sign of everything downstream, which is
exactly the kind of mistake that is hard to spot later.

In [38]:
def hamming(a, b):
    def to_bits(x):
        if isinstance(x, str):
            return np.unpackbits(np.frombuffer(bytes.fromhex(x), dtype=np.uint8))
        return np.asarray(x, dtype=np.uint8)
    return int(np.sum(to_bits(a) != to_bits(b)))


def similarity(a, b):
    """Fraction of agreeing bits, between 0 and 1."""
    return float(np.mean(np.asarray(a) == np.asarray(b)))

# ── test ──
try:
    x = np.array([1, 0, 1, 0], dtype=np.uint8)
    y = np.array([1, 1, 1, 0], dtype=np.uint8)
    ok  = check("hamming of identical arrays is 0", hamming(x, x) == 0)
    ok &= check("hamming x vs y is 1", hamming(x, y) == 1, f"got {hamming(x, y)}")
    ok &= check("similarity of identical arrays is 1.0", similarity(x, x) == 1.0)
    ok &= check("similarity x vs y is 0.75", abs(similarity(x, y) - 0.75) < 1e-9,
                f"got {similarity(x, y)}")
    ok &= check("the two are consistent",
                abs(similarity(x, y) - (1 - hamming(x, y) / len(x))) < 1e-9)
except NotImplementedError:
    print("not implemented yet")

PASS  hamming of identical arrays is 0
PASS  hamming x vs y is 1
PASS  similarity of identical arrays is 1.0
PASS  similarity x vs y is 0.75
PASS  the two are consistent


## 7. Does the hash track the geometry?

The point of all this: conformers that are geometrically close should get
hashes that are close.

`dm_rmsd(mol, i, j)` is the reference measure. Take the heavy-atom distance
list for each conformer, subtract them element by element, and take the root of
the mean square. No superposition is involved, which is the whole reason for
using it: the distances between atoms do not change when you rotate a molecule.

`evaluate(smiles, ...)` should hash a set of conformers, then correlate
DM-RMSD against hash similarity over every pair, and return the Spearman
coefficient with the sign flipped so that higher means better.

Why flipped: DM-RMSD is a distance and similarity is not, so when they track
each other properly the raw correlation comes out negative.

In [41]:
def dm_rmsd(mol, i, j):
    return float(np.sqrt(np.mean((features(mol, i) - features(mol, j)) ** 2)))

def evaluate(smiles, n_conf=30, n_bits=64, n_components=16):
    sc = fit_scaler(smiles)                      # fit_scaler
    pca = fit_pca(sc, smiles, n_components)                     # fit_pca
    m = make_conformers(smiles, n_conf, seed=TEST_SEED)                       # make_conformers, seed=TEST_SEED
    nc = m.GetNumConformers()
    Y = np.array([reduce(sc, pca, m, c) for c in range(nc)])                       # macierz po reduce, wiersz na konformer
    B = simhash(Y, n_bits)                      # simhash(Y, n_bits)
    pairs = [(i, j) for i in range(nc) for j in range(i + 1, nc)]
    dm = np.array([dm_rmsd(m, i, j) for i, j in pairs])
    sim = np.array([similarity(B[i], B[j]) for i, j in pairs])
    return -spearmanr(dm, sim).statistic if np.std(sim) > 0 else np.nan


# ── test ──
try:
    m = make_conformers(ASPIRIN, 5, seed=TEST_SEED)
    ok  = check("dm_rmsd of a conformer with itself is 0", dm_rmsd(m, 0, 0) == 0)
    ok &= check("dm_rmsd is symmetric", abs(dm_rmsd(m, 0, 1) - dm_rmsd(m, 1, 0)) < 1e-12)
    ok &= check("dm_rmsd of different conformers is positive", dm_rmsd(m, 0, 1) > 0)
    r_asp = evaluate(ASPIRIN)
    r_ibu = evaluate(IBUPROFEN)
    ok &= check("aspirin rho is a number", not np.isnan(r_asp), f"rho = {r_asp:.3f}")
    ok &= check("rho is positive, i.e. the sign is the right way round",
                r_asp > 0, f"rho = {r_asp:.3f}; negative means the sign is flipped")
    print(f"\n  aspirin   rho = {r_asp:+.3f}")
    print(f"  ibuprofen rho = {r_ibu:+.3f}")
except NotImplementedError:
    print("not implemented yet")

PASS  dm_rmsd of a conformer with itself is 0
PASS  dm_rmsd is symmetric
PASS  dm_rmsd of different conformers is positive


ValueError: X has 105 features, but StandardScaler is expecting 78 features as input.

## Done

Three things worth trying once everything passes, in rising order of interest:

**Drop the centring.** Fit the scaler with `StandardScaler(with_mean=False)`
and look at what `simhash` reports as the fraction of ones. This is the failure
that motivated the whole reference-ensemble design.

**Vary the bit budget.** Run `evaluate` at 16, 64, 256 and 1024 bits. The curve
flattens somewhere; find roughly where.

**Perturb a conformer.** Add gaussian noise of 0.001, 0.01 and 0.05 A to every
atom, hash the result, and count how many bits changed against the original.
This is the property that decides whether a hash can recognise a conformer or
merely rank it.
